## Add a tool

- An agent can only do what its tool list allows ; the loop never changes
- A tool is a function plus the one sentence the model reads about it
- Take one away and the agent has no way to finish ; add it and it does

Below is the harness from demos12 with three tools. It can list, read and run,
but not write. Run it, watch it fail to finish, then give it hands.

Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

A folder with a small program and a file of checks that fail.

In [ ]:
import shutil
from pathlib import Path

WORKSPACE = Path("workspace").resolve()

PROGRAM = """def line_total(price, quantity):
    return price + quantity


def invoice_total(lines):
    return sum(line_total(p, q) for p, q in lines)
"""

CHECKS = """from invoice import invoice_total

lines = [(10, 2), (5, 4)]      # two of something at 10, four of something at 5
got = invoice_total(lines)
assert got == 40, f"expected 40, got {got}"
print("all checks pass")
"""


def make_workspace():
    """Start again from a clean folder."""
    shutil.rmtree(WORKSPACE, ignore_errors=True)
    WORKSPACE.mkdir()
    (WORKSPACE / "invoice.py").write_text(PROGRAM)
    (WORKSPACE / "check.py").write_text(CHECKS)


make_workspace()

Three tools, and what the model is told about each.

In [ ]:
import subprocess
import sys


def _inside(path):
    """Resolve a path and refuse anything outside the workspace."""
    p = (WORKSPACE / path).resolve()
    if WORKSPACE not in p.parents:
        raise ValueError(f"{path} is outside the workspace")
    return p


def list_files():
    files = [p for p in sorted(WORKSPACE.rglob("*"))
             if p.is_file() and "__pycache__" not in p.parts]
    return "\n".join(p.relative_to(WORKSPACE).as_posix() for p in files)


def read_file(path):
    return _inside(path).read_text()


def run_python(path):
    r = subprocess.run([sys.executable, str(_inside(path))], cwd=WORKSPACE,
                       capture_output=True, text=True, timeout=30)
    return f"exit code {r.returncode}\n" + (r.stdout + r.stderr).strip()[-2000:]


SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files",
        "description": "List every file in the workspace.",
        "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {
        "name": "read_file",
        "description": "Return the contents of a file in the workspace.",
        "parameters": {"type": "object",
                       "properties": {"path": {"type": "string"}},
                       "required": ["path"]}}},
    {"type": "function", "function": {
        "name": "run_python",
        "description": "Run a Python file in the workspace and return its output and exit code.",
        "parameters": {"type": "object",
                       "properties": {"path": {"type": "string"}},
                       "required": ["path"]}}},
]

TOOLS = {"list_files": list_files, "read_file": read_file,
         "run_python": run_python}

The harness, exactly as in demos12.

In [ ]:
import json
from openai import OpenAI

client = OpenAI(base_url=BASE, api_key=KEY)

SYSTEM = ("You work in a small folder of files. Look before you act, check "
          "your work by running it, and finish with a short summary of what "
          "you did.")

TASK = ("The checks in check.py fail. Find out why, fix the program, and "
        "run the checks again to prove it.")


def brief(args):
    """Arguments, shortened enough to print on one line."""
    return ", ".join(f"{k}={str(v)[:30]!r}" for k, v in args.items())


def run_agent(task, max_turns=15, approve=None):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": task}]
    tokens = 0
    for turn in range(1, max_turns + 1):
        reply = client.chat.completions.create(model=MODEL, messages=messages,
                                               tools=SCHEMA, temperature=0,
                                               extra_body=THINKING)
        message = reply.choices[0].message
        tokens += reply.usage.total_tokens
        messages.append(message)
        print(f"turn {turn:2}  request {reply.usage.prompt_tokens:,} tokens")
        if not message.tool_calls:
            print(f"\n{turn} turns, {tokens:,} tokens in total\n")
            return message.content
        for call in message.tool_calls:
            name, args = call.function.name, json.loads(call.function.arguments)
            if approve and not approve(name, args):
                result = (f"{name} was refused by the person supervising you. "
                          "Do not try another route ; finish now and say what you could not do.")
            else:
                try:
                    result = TOOLS[name](**args)
                except Exception as e:
                    result = f"ERROR {type(e).__name__}: {e}"
            first = (result.splitlines() or [""])[0][:60]
            print(f"          {name}({brief(args)})  ->  {first}")
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": result})
    print(f"\nstopped after {max_turns} turns, {tokens:,} tokens in total\n")
    return None

Run it, with a limit of eight turns. It finds the bug in about three.
Watch what it does with the other five.

In [ ]:
make_workspace()
print(run_agent(TASK, max_turns=8) or "no answer")

### Exercise 1 : give it hands

The agent knew what to change by turn three and had no way to change it. It
did not say so. It guessed at files that do not exist until the limit stopped
it. An agent that is missing a tool does not fail cleanly ; it wanders.

1. Write `write_file(path, content)`. It should create or overwrite the file
   and return a short sentence saying what it did. `_inside(path)` keeps it in
   the workspace.
2. Write its entry for `SCHEMA` : a name, a sentence, and two string
   parameters, both required. Copy the shape of `read_file`'s entry.
3. Add it to `SCHEMA` and to `TOOLS`, then run the agent again.

The cell below is one answer. Write yours first, or run it as it is.

In [ ]:
def write_file(path, content):
    p = _inside(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"wrote {len(content)} characters to {path}"


WRITE_FILE = {"type": "function", "function": {
    "name": "write_file",
    "description": "Create or overwrite a file with the given content.",
    "parameters": {"type": "object",
                   "properties": {"path": {"type": "string"},
                                  "content": {"type": "string"}},
                   "required": ["path", "content"]}}}

SCHEMA.append(WRITE_FILE)
TOOLS["write_file"] = write_file

make_workspace()
print(run_agent(TASK))
print(run_python("check.py"))

Same loop, same task, same model. One entry in a list is the difference
between eight turns of guessing and a fix.

### Exercise 2 : take the meaning away

The model decides which tool to use from the sentence we wrote about it. Make
that sentence useless and see what happens.

1. Change `run_python`'s description to `"Does a thing."`.
2. Run the agent again. Does it still run the checks to prove its fix?
3. Put the description back.

In [ ]:
for entry in SCHEMA:
    if entry["function"]["name"] == "run_python":
        entry["function"]["description"] = "Does a thing."

make_workspace()
print(run_agent(TASK))

for entry in SCHEMA:
    if entry["function"]["name"] == "run_python":
        entry["function"]["description"] = ("Run a Python file in the workspace "
                                            "and return its output and exit code.")

### Try one of these

- Add `delete_file(path)`. Then give `run_agent` an `approve` function that
  refuses it, and see what the model says
- Rename `run_python` to `tool_3` in `SCHEMA` and `TOOLS` as well as blanking
  its description. The name alone was probably carrying the meaning
- Set `max_turns=2` and read what the model was in the middle of doing
- Change `TASK` to something the tools cannot do : *"email me the result"*